In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install torch  # torch is usually pre-installed

Mounted at /content/drive


In [ ]:
!git clone https://github.com/leon934/csce-636-proj.git
%cd csce-636-proj

Cloning into 'csce-636-proj'...
fatal: could not read Username for 'https://github.com': No such device or address
[Errno 2] No such file or directory: '636-project'
/content


In [ ]:
import sqlite3
import math
import numpy as np
import torch
from torch.utils.data import Dataset


class MHeightDataset(Dataset):
    def __init__(self, n, k, m, db_path, is_training=False):
        self.n = n
        self.k = k
        self.m = m
        self.is_training = is_training
        self.data = []

        table_name = f"n-{n}_k-{k}_m-{m}"
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()

        try:
            cursor.execute(f'SELECT p_matrix, m_height FROM "{table_name}"')
            for p_bytes, m_height in cursor.fetchall():
                p_array = np.frombuffer(p_bytes, dtype=np.int8).astype(np.float32)
                p_matrix = p_array.reshape(k, n - k)
                log_target = math.log2(max(1.0, float(m_height)))
                self.data.append((p_matrix, log_target))
        except sqlite3.OperationalError as e:
            print(f"Error reading from table {table_name}: {e}")
        finally:
            conn.close()

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        p_np, log_target = self.data[idx]
        P = torch.tensor(p_np, dtype=torch.float32)  # (k, n-k)

        if self.is_training:
            # Column permutations of P preserve m-height (relabeling parity positions
            # leaves the max-over-all-subsets definition of h_m unchanged).
            perm = torch.randperm(self.n - self.k)
            P = P[:, perm]
            # Negating a column of P preserves m-height (h_m depends only on |c_j|,
            # so scaling any column by -1 leaves every LP objective value unchanged).
            col_signs = torch.randint(0, 2, (1, self.n - self.k)).float() * 2 - 1
            P = P * col_signs

        y = torch.tensor([log_target], dtype=torch.float32)
        return P, y


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.02):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.bn1 = nn.BatchNorm1d(dim)
        self.fc2 = nn.Linear(dim, dim)
        self.bn2 = nn.BatchNorm1d(dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = F.relu(self.bn1(self.fc1(x)))
        out = self.dropout(out)
        out = self.bn2(self.fc2(out))
        return F.relu(out + x)


class DeepSetsNet(nn.Module):
    """
    Column-permutation-invariant network for estimating log2(m-height).

    Each column of P (a k-vector representing one parity position) is encoded
    independently by a shared MLP. The resulting embeddings are aggregated with
    sum + mean + max pooling, which is invariant to the order of the columns.
    Three scalar context values (n, k, m) are appended before the head.

    This architecture directly respects the mathematical symmetry that permuting
    the parity columns of G leaves the m-height of the code unchanged.
    """

    def __init__(self, k, n_minus_k, col_embed_dim=128, head_width=256, head_depth=4, dropout=0.02):
        super().__init__()

        # Shared encoder: maps each k-dimensional column → col_embed_dim embedding.
        # BatchNorm is applied over (batch * n_minus_k) samples, which is valid
        # because every column embedding is an independent forward pass through the same weights.
        self.col_encoder = nn.Sequential(
            nn.Linear(k, col_embed_dim),
            nn.BatchNorm1d(col_embed_dim),
            nn.ReLU(),
            nn.Linear(col_embed_dim, col_embed_dim),
            nn.BatchNorm1d(col_embed_dim),
            nn.ReLU(),
        )

        # sum + mean + max → 3 * col_embed_dim, then 3 context scalars (n, k, m)
        agg_dim = 3 * col_embed_dim + 3

        head_layers = [
            nn.Linear(agg_dim, head_width),
            nn.BatchNorm1d(head_width),
            nn.ReLU(),
        ]
        for _ in range(head_depth):
            head_layers.append(ResBlock(head_width, dropout))
        head_layers += [
            nn.Linear(head_width, head_width // 2),
            nn.ReLU(),
            nn.Linear(head_width // 2, 1),
        ]
        self.head = nn.Sequential(*head_layers)

    def forward(self, P, context):
        # P:       (batch, k, n_minus_k)
        # context: (batch, 3)  — [n, k, m] as floats
        batch, k, cols = P.shape

        # Flatten columns across the batch for the shared encoder
        P_cols = P.permute(0, 2, 1).reshape(batch * cols, k)  # (batch*cols, k)
        embeddings = self.col_encoder(P_cols)                  # (batch*cols, embed_dim)
        embeddings = embeddings.view(batch, cols, -1)          # (batch, cols, embed_dim)

        agg_sum  = embeddings.sum(dim=1)            # (batch, embed_dim)
        agg_mean = embeddings.mean(dim=1)           # (batch, embed_dim)
        agg_max  = embeddings.max(dim=1).values     # (batch, embed_dim)

        agg = torch.cat([agg_sum, agg_mean, agg_max, context], dim=1)
        return self.head(agg)


In [ ]:
import sys; sys.path.insert(0, 'ml_v3')
from train import train_combo
from datetime import datetime

sync_time = datetime.now().strftime("%Y%m%d_%H%M%S")
for n, k, m in [(9,4,2),(9,4,3),(9,4,4),(9,4,5),(9,5,2),(9,5,3),(9,5,4),(9,6,2),(9,6,3)]:
    train_combo(n, k, m, sync_time)